### Eco-Friendly Factory Production Scheduling
A manufacturing plant produces two types of sustainable packaging materials: Product A and Product B. The company wants to maximize its total profit over the next week based on production limits, resource constraints, and a strict environmental regulation.
1. Objective: Maximize the total net profit from producing Product A ($x_1$) and Product B ($x_2$).
* Each unit of Product A yields a profit of \\$50.
* Each unit of Product B yields a profit of \\$60.
2. Standard Constraints
* Resource Limit: Producing one unit of Product A requires 2 hours of machine time, and Product B requires 3 hours. The total available machine time for the week is 240 hours.
* Material Limit: Product A requires 4 kg of raw material per unit, and Product B requires 2 kg. The total available raw material is 300 kg.
* Non-Negativity: Production quantities cannot be negative ($x_1, x_2 \ge 0$).
3. The Either-Or Condition (Environmental Regulation)
To comply with local green energy policies, the factory must restrict its carbon footprint. The regulation states that the factory must satisfy at least one of the following two environmental conditions this week (it can satisfy both, but it must satisfy at least one)
* Condition 1 (Strict Carbon Cap): Total emissions cannot exceed 150 kg of CO2. Product A emits 3 kg per unit, and Product B emits 1 kg per unit.
$$3x_1 + x_2 \le 150$$
* Condition 2 (Clean Energy Offset): Alternatively, if they run higher emissions, they must heavily restrict the production of the more toxic Product A to a maximum of 25 units. $$x_1 \le 25$$

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>eco-friendly_scheduling</code.

In [13]:
from docplex.mp.model import Model

mdl = Model(name="eco-friendly_scheduling")

#### Define Parameters 

In [14]:
NET_PROFIT_PROD_A = 50
NET_PROFIT_PROD_B = 60

MAX_MACHINE_HRS = 240
MAX_MATERIALS = 300

CARBON_CAP = 150
CLEAN_ENERGY_OFFSET = 25

#### Define the Decision Variables
Continuous variables for the Products

In [15]:
x_A = mdl.continuous_var(name="Product_A_Units", lb=0)
x_B = mdl.continuous_var(name="Product_B_Units", lb=0)

Binary variable for environmental compliance

In [16]:
y = mdl.binary_var(name="Environmental_Compliance")

#### Define the Constraints
Global Constraints

In [17]:
mdl.add_constraint(2 * x_A + 3 * x_B <= MAX_MACHINE_HRS, "Max_Machine_Hrs")
mdl.add_constraint(4 * x_A + 2 * x_B <= MAX_MATERIALS, "Max_Materials")

docplex.mp.LinearConstraint[Max_Materials](4Product_A_Units+2Product_B_Units,LE,300)

Either-Or Constraints using Indicators

In [18]:
# Strict Carbon Cap
mdl.add_indicator(
    y, 
    (3 * x_A) + x_B <= MAX_EMISSIONS, 
    active_value=1,
    name="Strict_Carbon_Cap"
)

# Clean Energy Offset
mdl.add_indicator(
    y, 
    x_A <= CLEAN_ENERGY_OFFSET, 
    active_value=0, 
    name="Clean_Energy_Offset"
)

docplex.mp.constr.IndicatorConstraint[Clean_Energy_Offset](Environmental_Compliance,Product_A_Units<=25,true=0)

#### Define the Objective Function

In [19]:
total_profit = (NET_PROFIT_PROD_A * x_A) + (NET_PROFIT_PROD_B * x_B)
mdl.maximize(total_profit)

#### Solve the Model

In [20]:
print("Solving model...")
solution = mdl.solve(log_output=True)

Solving model...
Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 5 rows, 4 columns, and 11 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing fixed 0 vars, tightened 2 bounds.
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve modified 3 coefficients.
Reduced MIP has 5 rows, 4 columns, and 11 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 10 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best 

#### Print the Solution

In [24]:
if solution:
    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Status: {mdl.get_solve_status()}")
    print(f"Total Net Profit: ${solution.objective_value:,.2f}")
    print(f"Product A: {solution[x_A]:.2f} units")
    print(f"Product B: {solution[x_B]:.2f} units")

    machine_hours = 2 * solution[x_A] + 3 * solution[x_B]
    resources_used = 4 * solution[x_A] + 2 * solution[x_B]
    print(f"Machine Hours Used: ${machine_hours:,} / ${MAX_MACHINE_HRS:,}")
    print(f"Resources Used: {resources_used:.0f} / {MAX_MATERIALS:,}kg")
    
    if solution[y] == 1:
        print("Selected Regulation: Strict Carbon Cap")
    else:
        print("Selected Regulation: Clean Energy Offset")
else:
    print("\nNo optimal solution found. Check model constraints for infeasibility.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Net Profit: $5,100.00
Product A: 30.00 units
Product B: 60.00 units
Machine Hours Used: $240.0 / $240
Resources Used: 240 / 300kg
Selected Regulation: Strict Carbon Cap


### Urban Tech Office Expansion
A tech company is expanding its regional headquarters and needs to determine how many Workstations and Meeting Pods to install. The company wants to maximize the total employee capacity of the new space.

1. Objective Function: Maximize the total capacity (Z), where a Workstation accommodates 1 employee and a Meeting Pod accommodates 4 employees:
$$\text{maximize} \; Z = 1x_1 + 4x_2$$ 
2. Standard Constraints
* Budget Limit: Each Workstation costs \\$500 and each Pod costs\\$2,000. Total budget is \\$40,000.
$$500x_1 + 2000x_2 \le 40000$$ 
* Floor Space: Each Workstation takes up 2 square meters and each Pod takes up 6 square meters. Total available space is 150 square meters.
$$2x_1 + 6x_2 \le 150$$ 
3. The IF-THEN Condition (IT Infrastructure Regulation)
The office building's server room has a specific power threshold. The facility manager introduces the following logical rule:
IF the company installs more than 12 Meeting Pods ($x_2 > 12$),
THEN they must limit the number of Workstations to at most 15 ($x_1 \leq 15$) to avoid overloading the local electrical circuit.

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>tech_office_expansion</code>.
nsion</code>.ectrical circuit.

In [25]:
from docplex.mp.model import Model

mdl = Model(name="tech_office_expansion")

#### Define Parameters 

In [26]:
WORKSTATION_CAP = 1
MEETING_POD_CAP = 4

BUDGET_LIMIT = 40000
FLOOR_SPACE_LIMIT = 150

POD_LIMIT = 12
WORKSTATION_LIMIT = 15

#### Define the Decision Variables
Continuous variables for the Products

In [27]:
x_1 = mdl.continuous_var(name="Workstation_Units", lb=0)
x_2 = mdl.continuous_var(name="Meeting_Pods", lb=0)

Binary variable for environmental compliance

In [28]:
y = mdl.binary_var(name="Pods_Exceed_12")

#### Define the Constraints
Global Constraints

In [29]:
mdl.add_constraint(500 * x_1 + 2000 * x_2 <= BUDGET_LIMIT, "Budget_Limit")
mdl.add_constraint(2 * x_1 + 6 * x_2 <= FLOOR_SPACE_LIMIT, "Floor_Space_Limit")

docplex.mp.LinearConstraint[Floor_Space_Limit](2Workstation_Units+6Meeting_Pods,LE,150)

IF-THEN Constraint: IF the company installs more than 12 Meeting Pods ($x_2 > 12$),
THEN they must limit the number of Workstations to at most 15 ($x_1 \leq 15$) to avoid overloading the local electrical circuit.
* if (y=0), then $x_2 \leq 12$
* if (y=1), then $x_1 \leq 15$

In [30]:
mdl.add_indicator(
    y, 
    x_2 <= POD_LIMIT, 
    active_value=0,
    name="Link_y_zero"
)

mdl.add_indicator(
    y, 
    x_1 <= WORKSTATION_LIMIT, 
    active_value=1, 
    name="Circuit_Overload_Restriction"
)

docplex.mp.constr.IndicatorConstraint[Circuit_Overload_Restriction](Pods_Exceed_12,Workstation_Units<=15,true=1)

#### Define the Objective Function

In [31]:
total_capacity = (WORKSTATION_CAP * x_1) + (MEETING_POD_CAP * x_2)
mdl.maximize(total_capacity)

#### Solve the Model

In [32]:
print("Solving model...")
solution = mdl.solve(log_output=True)

Solving model...
Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 4 rows, 3 columns, and 8 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 1 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 4 rows, 3 columns, and 8 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 1 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 10 threads.
Root relaxation solution time = 0.00 sec. (0.00 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                            0

#### Print the Solution

In [41]:
if solution:
    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Status: {mdl.get_solve_status()}")
    print(f"Total Capacity: {solution.objective_value:.0f} employees")
    print(f"Workstations: {solution[x_1]:.0f} units")
    print(f"Meeting Pods: {solution[x_2]:.0f} units")
    
    # Verify budget and space usage
    budget_used = 500 * solution[x_1] + 2000 * solution[x_2]
    space_used = 2 * solution[x_1] + 6 * solution[x_2]
    print(f"Budget Used: ${budget_used:,.0f} / ${BUDGET_LIMIT:,}")
    print(f"Floor Space Used: {space_used:.0f} / {FLOOR_SPACE_LIMIT} sq meters")
else:
    print("\nNo optimal solution found. Check constraints for infeasibility.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Capacity: 80 employees
Workstations: 0 units
Meeting Pods: 20 units
Budget Used: $40,000 / $40,000
Floor Space Used: 120 / 150 sq meters
